# Training Notebook (Spark ML, Distributed)

This notebook trains Spark ML baselines on pre-engineered features and saves outputs to both HDFS and local mounted folders.

Main outputs:
- HDFS metrics/predictions by run id
- Local metrics and preview CSV under /workspace/code/hybrid_regression_holt_winters/results/training_sparkml
- Local best model pipeline under /workspace/code/hybrid_regression_holt_winters/models/training_sparkml


In [1]:
import json
import os
import urllib.request
import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .appName("DemandPredictionTrainingSparkML")
    .master("spark://master:7077")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def cluster_util(stage_name):
    print(f"\n===== CLUSTER UTILIZATION: {stage_name} =====")
    try:
        payload = json.load(urllib.request.urlopen("http://master:8080/json/"))
        workers = payload.get("workers", [])
        print("alive workers:", payload.get("aliveworkers"))
        print("active apps :", len(payload.get("activeapps", [])))
        for w in workers:
            print("worker", w.get("id"), "cores", f"{w.get('coresused', 0)}/{w.get('cores', 0)}", "memory", f"{w.get('memoryused', 0)}/{w.get('memory', 0)}")
    except Exception as e:
        print("Could not query Spark Master JSON:", e)

cluster_util("session_started")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fc84190c-0053-41d7-b5dd-5c6ae6561eeb;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 121ms :: artifacts dl 5ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------


===== CLUSTER UTILIZATION: session_started =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


In [2]:
FEATURE_PATH = "/user/data/feature_engineering/demand_prediction_features"
OUT_BASE_ROOT = "/user/data/results/demand_prediction/sparkml"
TARGET_COL = "pickup_demand"

LOCAL_BASE = "/workspace/code/hybrid_regression_holt_winters"
LOCAL_RESULTS_ROOT = f"{LOCAL_BASE}/results/training_sparkml"
LOCAL_MODELS_ROOT = f"{LOCAL_BASE}/models/training_sparkml"
LOCAL_ARTIFACTS_ROOT = f"{LOCAL_BASE}/artifacts/training_sparkml"
os.makedirs(LOCAL_RESULTS_ROOT, exist_ok=True)
os.makedirs(LOCAL_MODELS_ROOT, exist_ok=True)
os.makedirs(LOCAL_ARTIFACTS_ROOT, exist_ok=True)

# Optional manual override; keep None to auto-detect from data.
TRAIN_START_TS = None
TRAIN_END_TS = None

feature_cols = [
    "hour", "dow", "month", "is_weekend",
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12", "lag_144", "lag_1008",
    "roll_mean_6", "roll_mean_18", "roll_std_18",
]

df_all = spark.read.parquet(FEATURE_PATH)
if "pickup_bin_10m" not in df_all.columns:
    raise ValueError("Missing required column: pickup_bin_10m")

available_bounds = df_all.agg(
    F.min("pickup_bin_10m").alias("min_ts"),
    F.max("pickup_bin_10m").alias("max_ts")
).first()

if available_bounds["min_ts"] is None or available_bounds["max_ts"] is None:
    raise ValueError("Feature dataset has no usable pickup_bin_10m timestamps.")

effective_start = TRAIN_START_TS if TRAIN_START_TS is not None else available_bounds["min_ts"]
effective_end = TRAIN_END_TS if TRAIN_END_TS is not None else available_bounds["max_ts"]

window_cond = F.col("pickup_bin_10m") >= F.to_timestamp(F.lit(effective_start))
if TRAIN_END_TS is not None:
    window_cond = window_cond & (F.col("pickup_bin_10m") < F.to_timestamp(F.lit(effective_end)))
else:
    window_cond = window_cond & (F.col("pickup_bin_10m") <= F.to_timestamp(F.lit(effective_end)))

df = df_all.where(window_cond).cache()

rows = df.count()
if rows == 0:
    raise ValueError(
        f"No rows found in requested training window [{effective_start}, {effective_end}]. "
        f"Available range is [{available_bounds['min_ts']}, {available_bounds['max_ts']}]."
    )

run_id = f"{pd.Timestamp(effective_start):%Y%m%d_%H%M%S}_{pd.Timestamp(effective_end):%Y%m%d_%H%M%S}"
OUT_BASE = f"{OUT_BASE_ROOT}/{run_id}"
LOCAL_RESULTS_DIR = f"{LOCAL_RESULTS_ROOT}/{run_id}"
LOCAL_MODELS_DIR = f"{LOCAL_MODELS_ROOT}/{run_id}"
LOCAL_ARTIFACTS_DIR = f"{LOCAL_ARTIFACTS_ROOT}/{run_id}"
os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
os.makedirs(LOCAL_ARTIFACTS_DIR, exist_ok=True)

print(f"Feature rows in effective training window [{effective_start}, {effective_end}]:", rows)
df.groupBy("split").count().show()
cluster_util("after_feature_read")

# Rebuild split inside the filtered window to avoid empty train/test partitions.
bounds = df.agg(F.min("pickup_bin_10m").alias("min_ts"), F.max("pickup_bin_10m").alias("max_ts")).first()
if bounds["min_ts"] is None or bounds["max_ts"] is None:
    raise ValueError("Cannot compute local split bounds.")

min_epoch = int(bounds["min_ts"].timestamp())
max_epoch = int(bounds["max_ts"].timestamp())
cut_epoch = int(min_epoch + 0.7 * (max_epoch - min_epoch))

df = df.withColumn(
    "split_local",
    F.when(
        F.col("pickup_bin_10m") < F.to_timestamp(F.from_unixtime(F.lit(cut_epoch))),
        F.lit("train")
    ).otherwise(F.lit("test"))
).cache()

train_df = df.where(F.col("split_local") == "train")
test_df = df.where(F.col("split_local") == "test")
print("Train rows:", train_df.count())
print("Test rows :", test_df.count())

Feature rows in effective training window [2025-10-07 22:50:00, 2025-11-30 23:50:00]: 2039146


+-----+-------+
|split|  count|
+-----+-------+
|train|1427376|
| test| 611770|
+-----+-------+


===== CLUSTER UTILIZATION: after_feature_read =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


Train rows: 1427376
Test rows : 611770


In [3]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

# Resource-safe defaults for this cluster; tune upward when cluster RAM/CPU is larger.
FAST_MODE = False
TRAIN_SAMPLE_FRACTION = 0.08 if FAST_MODE else 1.0

train_fit_df = train_df
if TRAIN_SAMPLE_FRACTION < 1.0:
    train_fit_df = train_df.sample(withReplacement=False, fraction=TRAIN_SAMPLE_FRACTION, seed=42).cache()
    print(f"[FAST_MODE] Using sampled train rows: {train_fit_df.count()} / {train_df.count()}")

mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")

metrics_rows = []
pred_union = None
fitted_models = {}

def collect_metrics_and_predictions(model_name, pred_df):
    global pred_union
    mae_val = float(mae_eval.evaluate(pred_df))
    rmse_val = float(rmse_eval.evaluate(pred_df))
    mape_val = (
        pred_df.where(F.col(TARGET_COL) > 0)
        .select((F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)).alias("ape"))
        .agg((F.avg(F.col("ape")) * 100.0).alias("mape"))
        .first()["mape"]
    )

    metrics_rows.append({
        "model": model_name,
        "MAE": mae_val,
        "RMSE": rmse_val,
        "MAPE": float(mape_val) if mape_val is not None else None,
    })

    slim_pred = pred_df.select(
        F.lit(model_name).alias("model"),
        "PULocationID",
        "pickup_bin_10m",
        TARGET_COL,
        "prediction",
    )
    pred_union = slim_pred if pred_union is None else pred_union.unionByName(slim_pred)

In [4]:
# Model 1: Linear Regression
model_name = "linear_regression"
cluster_util(f"before_train_{model_name}")

lr = LinearRegression(
    featuresCol="features",
    labelCol=TARGET_COL,
    predictionCol="prediction",
    maxIter=60 if FAST_MODE else 100,
    regParam=0.1,
    elasticNetParam=0.0,
)

lr_pipeline = Pipeline(stages=[assembler, lr])
lr_fitted = lr_pipeline.fit(train_fit_df)
fitted_models[model_name] = lr_fitted

lr_pred = lr_fitted.transform(test_df).withColumn(
    "prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction"))
)
collect_metrics_and_predictions(model_name, lr_pred)
cluster_util(f"after_train_{model_name}")
print("Finished:", model_name)


===== CLUSTER UTILIZATION: before_train_linear_regression =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


26/04/04 06:06:16 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/04 06:06:16 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/04/04 06:06:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



===== CLUSTER UTILIZATION: after_train_linear_regression =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048
Finished: linear_regression


In [5]:
# Model 2: Random Forest (reduced params for stability)
model_name = "random_forest"
cluster_util(f"before_train_{model_name}")

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    predictionCol="prediction",
    numTrees=30 if FAST_MODE else 80,
    maxDepth=7 if FAST_MODE else 10,
    maxBins=32,
    seed=42,
)

rf_pipeline = Pipeline(stages=[assembler, rf])
rf_fitted = rf_pipeline.fit(train_fit_df)
fitted_models[model_name] = rf_fitted

rf_pred = rf_fitted.transform(test_df).withColumn(
    "prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction"))
)
collect_metrics_and_predictions(model_name, rf_pred)
cluster_util(f"after_train_{model_name}")
print("Finished:", model_name)


===== CLUSTER UTILIZATION: before_train_random_forest =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


26/04/04 06:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1678.0 KiB
26/04/04 06:06:44 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/04/04 06:06:52 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/04/04 06:07:01 WARN DAGScheduler: Broadcasting large task binary with size 1953.2 KiB
26/04/04 06:07:03 WARN DAGScheduler: Broadcasting large task binary with size 12.3 MiB
26/04/04 06:07:15 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB



===== CLUSTER UTILIZATION: after_train_random_forest =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048
Finished: random_forest


In [6]:
# Model 3: GBT (reduced params for stability)
model_name = "gbt"
cluster_util(f"before_train_{model_name}")

gbt = GBTRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    predictionCol="prediction",
    maxIter=20 if FAST_MODE else 60,
    maxDepth=3 if FAST_MODE else 5,
    stepSize=0.05,
    subsamplingRate=0.7,
    seed=42,
)

gbt_pipeline = Pipeline(stages=[assembler, gbt])
gbt_fitted = gbt_pipeline.fit(train_fit_df)
fitted_models[model_name] = gbt_fitted

gbt_pred = gbt_fitted.transform(test_df).withColumn(
    "prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction"))
)
collect_metrics_and_predictions(model_name, gbt_pred)
cluster_util(f"after_train_{model_name}")
print("Finished:", model_name)


===== CLUSTER UTILIZATION: before_train_gbt =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


26/04/04 06:07:34 WARN TransportChannelHandler: Exception in connection from /172.18.0.2:53610
java.io.IOException: Connection reset by peer
	at java.base/sun.nio.ch.FileDispatcherImpl.read0(Native Method)
	at java.base/sun.nio.ch.SocketDispatcher.read(SocketDispatcher.java:39)
	at java.base/sun.nio.ch.IOUtil.readIntoNativeBuffer(IOUtil.java:276)
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:233)
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:223)
	at java.base/sun.nio.ch.SocketChannelImpl.read(SocketChannelImpl.java:356)
	at io.netty.buffer.PooledByteBuf.setBytes(PooledByteBuf.java:254)
	at io.netty.buffer.AbstractByteBuf.writeBytes(AbstractByteBuf.java:1132)
	at io.netty.channel.socket.nio.NioSocketChannel.doReadBytes(NioSocketChannel.java:357)
	at io.netty.channel.nio.AbstractNioByteChannel$NioByteUnsafe.read(AbstractNioByteChannel.java:151)
	at io.netty.channel.nio.NioEventLoop.processSelectedKey(NioEventLoop.java:788)
	at io.netty.channel.nio.NioEventLoop.processSelectedKeysO


===== CLUSTER UTILIZATION: after_train_gbt =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048
Finished: gbt


In [7]:
metrics_pdf = pd.DataFrame(metrics_rows).sort_values("MAPE")
display(metrics_pdf)

if len(metrics_pdf) == 0:
    raise ValueError("No model finished successfully. Check Spark logs/resources.")

best_model_name = metrics_pdf.iloc[0]["model"]
print("Best model by MAPE:", best_model_name)

,model,MAE,RMSE,MAPE
1,random_forest,1.139012,3.225078,49.499304
2,gbt,1.161020,3.254407,50.665323
0,linear_regression,1.073391,2.658029,51.450893


Best model by MAPE: random_forest


In [8]:
metrics_sdf = spark.createDataFrame(metrics_rows)
metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
pred_union.write.mode("overwrite").partitionBy("model").parquet(f"{OUT_BASE}/predictions")

best_model_path = f"{LOCAL_MODELS_DIR}/best_model_pipeline"
fitted_models[best_model_name].write().overwrite().save(best_model_path)

metrics_csv_path = f"{LOCAL_RESULTS_DIR}/metrics.csv"
metrics_json_path = f"{LOCAL_RESULTS_DIR}/metrics.json"
best_pred_csv_path = f"{LOCAL_ARTIFACTS_DIR}/best_model_predictions_preview.csv"
run_meta_path = f"{LOCAL_RESULTS_DIR}/run_metadata.json"

metrics_pdf.to_csv(metrics_csv_path, index=False)
metrics_pdf.to_json(metrics_json_path, orient="records", indent=2)

best_pred_pdf = (
    pred_union.where(F.col("model") == best_model_name)
    .orderBy("pickup_bin_10m", "PULocationID")
    .limit(5000)
    .toPandas()
)
best_pred_pdf.to_csv(best_pred_csv_path, index=False)

run_meta = {
    "run_id": run_id,
    "feature_path": FEATURE_PATH,
    "effective_start": str(effective_start),
    "effective_end": str(effective_end),
    "hdfs_output_base": OUT_BASE,
    "best_model": best_model_name,
    "best_model_local_path": best_model_path,
}
with open(run_meta_path, "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)

print("Saved HDFS outputs:")
print(f"- {OUT_BASE}/metrics")
print(f"- {OUT_BASE}/predictions")
print("Saved local outputs:")
print("-", metrics_csv_path)
print("-", metrics_json_path)
print("-", best_pred_csv_path)
print("-", best_model_path)
print("-", run_meta_path)
metrics_sdf.orderBy(F.col("MAPE").asc_nulls_last()).show(truncate=False)
cluster_util("after_output_write")

26/04/04 06:08:35 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.2: Command exited with code 137
26/04/04 06:08:35 WARN TaskSetManager: Lost task 0.0 in stage 674.0 (TID 7435) (172.18.0.2 executor 3): ExecutorLostFailure (executor 3 exited caused by one of the running tasks) Reason: Command exited with code 137
26/04/04 06:08:35 WARN TaskSetManager: Lost task 3.0 in stage 674.0 (TID 7438) (172.18.0.2 executor 3): ExecutorLostFailure (executor 3 exited caused by one of the running tasks) Reason: Command exited with code 137
26/04/04 06:08:35 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_13_2 !
26/04/04 06:08:35 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_43_2 !
26/04/04 06:08:35 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_13_5 !
26/04/04 06:08:35 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_43_5 !


Saved HDFS outputs:
- /user/data/results/demand_prediction/sparkml/20251007_225000_20251130_235000/metrics
- /user/data/results/demand_prediction/sparkml/20251007_225000_20251130_235000/predictions
Saved local outputs:
- /workspace/code/hybrid_regression_holt_winters/results/training_sparkml/20251007_225000_20251130_235000/metrics.csv
- /workspace/code/hybrid_regression_holt_winters/results/training_sparkml/20251007_225000_20251130_235000/metrics.json
- /workspace/code/hybrid_regression_holt_winters/artifacts/training_sparkml/20251007_225000_20251130_235000/best_model_predictions_preview.csv
- /workspace/code/hybrid_regression_holt_winters/models/training_sparkml/20251007_225000_20251130_235000/best_model_pipeline
- /workspace/code/hybrid_regression_holt_winters/results/training_sparkml/20251007_225000_20251130_235000/run_metadata.json
+------------------+------------------+-----------------+-----------------+
|MAE               |MAPE              |RMSE             |model            |


In [9]:
# Optional cleanup: run this when training is done.
spark.catalog.clearCache()
spark.stop()
print("Training Spark session stopped.")

Training Spark session stopped.
